# Day 13 — Advanced SQL
### Window Functions · RANK · ROW_NUMBER · LAG · LEAD · CTEs

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import duckdb

# Load Titanic & create DuckDB connection
df = pd.read_csv(r"C:\DS-AI-75d\titanic.csv")
con = duckdb.connect()
con.register("titanic", df)


def sql(query):
    return con.execute(query).df()


print(f"Pandas:  {pd.__version__}")
print(f"DuckDB:  {duckdb.__version__}")
print(f"Dataset: {df.shape}")
print("Ready! ✅")

Pandas:  2.3.3
DuckDB:  1.5.1
Dataset: (891, 12)
Ready! ✅


## 2. What are Window Functions?

In [ ]:
print("=" * 55)
print("       WHAT ARE WINDOW FUNCTIONS?")
print("=" * 55)
print(
    """
Window functions perform calculations across a set of
rows RELATED to the current row — without collapsing
them into a single result like GROUP BY does.

GROUP BY:  891 rows → 3 rows (one per class)
WINDOW:    891 rows → 891 rows (keeps all rows!)

SYNTAX:
  function() OVER (
      PARTITION BY column   -- like GROUP BY
      ORDER BY column       -- defines row order
  )

COMMON WINDOW FUNCTIONS:
  ROW_NUMBER()  — unique sequential number (1,2,3...)
  RANK()        — rank with gaps (1,2,2,4...)
  DENSE_RANK()  — rank without gaps (1,2,2,3...)
  LAG(col, n)   — value from n rows BEFORE current row
  LEAD(col, n)  — value from n rows AFTER current row
  SUM() OVER()  — running total
  AVG() OVER()  — moving average
"""
)

       WHAT ARE WINDOW FUNCTIONS?

Window functions perform calculations across a set of
rows RELATED to the current row — without collapsing
them into a single result like GROUP BY does.

GROUP BY:  891 rows → 3 rows (one per class)
WINDOW:    891 rows → 891 rows (keeps all rows!)

SYNTAX:
  function() OVER (
      PARTITION BY column   -- like GROUP BY
      ORDER BY column       -- defines row order
  )

COMMON WINDOW FUNCTIONS:
  ROW_NUMBER()  — unique sequential number (1,2,3...)
  RANK()        — rank with gaps (1,2,2,4...)
  DENSE_RANK()  — rank without gaps (1,2,2,3...)
  LAG(col, n)   — value from n rows BEFORE current row
  LEAD(col, n)  — value from n rows AFTER current row
  SUM() OVER()  — running total
  AVG() OVER()  — moving average



## 3. ROW_NUMBER, RANK & DENSE_RANK

In [ ]:
print("=" * 55)
print("    ROW_NUMBER vs RANK vs DENSE_RANK")
print("=" * 55)

# Show difference between ranking functions
print("\n--- Q1: Rank passengers by fare within each class ---")
print(
    sql(
        """
    SELECT 
        Name,
        Pclass,
        Fare,
        ROW_NUMBER() OVER (PARTITION BY Pclass ORDER BY Fare DESC) as row_num,
        RANK()       OVER (PARTITION BY Pclass ORDER BY Fare DESC) as rank,
        DENSE_RANK() OVER (PARTITION BY Pclass ORDER BY Fare DESC) as dense_rank
    FROM titanic
    WHERE Pclass = 1
    ORDER BY Fare DESC
    LIMIT 10
"""
    )
)

# Top fare payer per class
print("\n--- Q2: Highest fare payer in each class ---")
print(
    sql(
        """
    SELECT Pclass, Name, Fare, Survived
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY Pclass ORDER BY Fare DESC) as rn
        FROM titanic
    )
    WHERE rn = 1
    ORDER BY Pclass
"""
    )
)

# Youngest survivor per class
print("\n--- Q3: Youngest survivor in each class ---")
print(
    sql(
        """
    SELECT Pclass, Name, Age, Fare, Sex
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY Pclass 
                ORDER BY Age ASC
            ) as age_rank
        FROM titanic
        WHERE Survived = 1 AND Age IS NOT NULL
    )
    WHERE age_rank = 1
    ORDER BY Pclass
"""
    )
)

    ROW_NUMBER vs RANK vs DENSE_RANK

--- Q1: Rank passengers by fare within each class ---
                                              Name  Pclass      Fare  row_num  \
0                           Lesurer, Mr. Gustave J       1  512.3292        1   
1               Cardeza, Mr. Thomas Drake Martinez       1  512.3292        2   
2                                 Ward, Miss. Anna       1  512.3292        3   
3                   Fortune, Mr. Charles Alexander       1  263.0000        7   
4                       Fortune, Miss. Mabel Helen       1  263.0000        6   
5                   Fortune, Miss. Alice Elizabeth       1  263.0000        5   
6                                Fortune, Mr. Mark       1  263.0000        4   
7                       Ryerson, Miss. Emily Borie       1  262.3750        8   
8            Ryerson, Miss. Susan Parker "Suzette"       1  262.3750        9   
9  Baxter, Mrs. James (Helene DeLaudeniere Chaput)       1  247.5208       10   

   rank  dense_r

## 4. LAG & LEAD Functions

In [ ]:
print("=" * 55)
print("           LAG & LEAD FUNCTIONS")
print("=" * 55)

# Create age-ordered dataset to show LAG/LEAD
print("\n--- Q4: Age differences between consecutive passengers ---")
print(
    sql(
        """
    SELECT 
        Name,
        Age,
        LAG(Age, 1)  OVER (ORDER BY Age) as prev_age,
        LEAD(Age, 1) OVER (ORDER BY Age) as next_age,
        Age - LAG(Age, 1) OVER (ORDER BY Age) as diff_from_prev
    FROM titanic
    WHERE Age IS NOT NULL
    ORDER BY Age
    LIMIT 10
"""
    )
)

# LAG within partition — fare comparison within class
print("\n--- Q5: Each passenger's fare vs previous passenger in same class ---")
print(
    sql(
        """
    SELECT 
        Name,
        Pclass,
        Fare,
        LAG(Fare) OVER (PARTITION BY Pclass ORDER BY Fare) as prev_fare,
        ROUND(Fare - LAG(Fare) OVER (PARTITION BY Pclass ORDER BY Fare), 2) as fare_diff
    FROM titanic
    WHERE Pclass = 1
    ORDER BY Fare DESC
    LIMIT 8
"""
    )
)

# Running total with SUM OVER
print("\n--- Q6: Running total of fares collected by class ---")
print(
    sql(
        """
    SELECT 
        PassengerId,
        Pclass,
        Fare,
        ROUND(SUM(Fare) OVER (
            PARTITION BY Pclass 
            ORDER BY PassengerId
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2) as running_total_fare
    FROM titanic
    WHERE Pclass = 1
    ORDER BY PassengerId
    LIMIT 8
"""
    )
)

           LAG & LEAD FUNCTIONS

--- Q4: Age differences between consecutive passengers ---
                              Name   Age  prev_age  next_age  diff_from_prev
0  Thomas, Master. Assad Alexander  0.42       NaN      0.67             NaN
1        Hamalainen, Master. Viljo  0.67      0.42      0.75            0.25
2    Baclini, Miss. Helene Barbara  0.75      0.67      0.75            0.08
3           Baclini, Miss. Eugenie  0.75      0.75      0.83            0.00
4    Caldwell, Master. Alden Gates  0.83      0.75      0.83            0.08
5  Richards, Master. George Sibley  0.83      0.83      0.92            0.00
6   Allison, Master. Hudson Trevor  0.92      0.83      1.00            0.09
7     Panula, Master. Eino Viljami  1.00      0.92      1.00            0.08
8     Johnson, Miss. Eleanor Ileen  1.00      1.00      1.00            0.00
9        Becker, Master. Richard F  1.00      1.00      1.00            0.00

--- Q5: Each passenger's fare vs previous passenger in same 

## 5. CTEs — Common Table Expressions

In [ ]:
print("=" * 55)
print("    CTEs — COMMON TABLE EXPRESSIONS")
print("=" * 55)
print(
    """
A CTE is a named temporary result set you can reference
in the same query — like creating a temp variable in SQL.

SYNTAX:
  WITH cte_name AS (
      SELECT ...
  )
  SELECT * FROM cte_name

BENEFITS:
  - Makes complex queries readable
  - Can reference the same CTE multiple times
  - Chain multiple CTEs together
  - Much cleaner than nested subqueries
"""
)

# Single CTE
print("\n--- Q7: CTE — Survival stats then filter ---")
print(
    sql(
        """
    WITH survival_stats AS (
        SELECT 
            Pclass,
            Sex,
            COUNT(*) as total,
            SUM(Survived) as survived,
            ROUND(AVG(Survived)*100, 1) as survival_pct,
            ROUND(AVG(Fare), 2) as avg_fare
        FROM titanic
        GROUP BY Pclass, Sex
    )
    SELECT *
    FROM survival_stats
    WHERE survival_pct > 50
    ORDER BY survival_pct DESC
"""
    )
)

# Multiple CTEs chained
print("\n--- Q8: Multiple CTEs chained together ---")
print(
    sql(
        """
    WITH 
    class_avg AS (
        SELECT Pclass, ROUND(AVG(Fare), 2) as avg_fare
        FROM titanic
        GROUP BY Pclass
    ),
    above_avg AS (
        SELECT t.Name, t.Pclass, t.Fare, t.Survived,
               c.avg_fare,
               ROUND(t.Fare - c.avg_fare, 2) as fare_above_avg
        FROM titanic t
        JOIN class_avg c ON t.Pclass = c.Pclass
        WHERE t.Fare > c.avg_fare
    )
    SELECT Pclass, COUNT(*) as passengers,
           ROUND(AVG(Survived)*100,1) as survival_pct
    FROM above_avg
    GROUP BY Pclass
    ORDER BY Pclass
"""
    )
)

    CTEs — COMMON TABLE EXPRESSIONS

A CTE is a named temporary result set you can reference
in the same query — like creating a temp variable in SQL.

SYNTAX:
  WITH cte_name AS (
      SELECT ...
  )
  SELECT * FROM cte_name

BENEFITS:
  - Makes complex queries readable
  - Can reference the same CTE multiple times
  - Chain multiple CTEs together
  - Much cleaner than nested subqueries


--- Q7: CTE — Survival stats then filter ---
   Pclass     Sex  total  survived  survival_pct  avg_fare
0       1  female     94      91.0          96.8    106.13
1       2  female     76      70.0          92.1     21.97

--- Q8: Multiple CTEs chained together ---
   Pclass  passengers  survival_pct
0       1          66          77.3
1       2          80          56.3
2       3         152          27.0


## 6. Advanced Window Functions

In [ ]:
print("=" * 55)
print("      ADVANCED WINDOW FUNCTIONS")
print("=" * 55)

# Percentile/NTILE
print("\n--- Q9: Divide passengers into fare quartiles ---")
print(
    sql(
        """
    WITH fare_quartiles AS (
        SELECT 
            Name, Pclass, Fare, Survived,
            NTILE(4) OVER (ORDER BY Fare) as fare_quartile
        FROM titanic
    )
    SELECT 
        fare_quartile,
        COUNT(*) as passengers,
        ROUND(MIN(Fare), 2) as min_fare,
        ROUND(MAX(Fare), 2) as max_fare,
        ROUND(AVG(Survived)*100, 1) as survival_pct
    FROM fare_quartiles
    GROUP BY fare_quartile
    ORDER BY fare_quartile
"""
    )
)

# Moving average
print("\n--- Q10: Running avg fare for 1st class passengers ---")
print(
    sql(
        """
    SELECT 
        PassengerId,
        Fare,
        ROUND(AVG(Fare) OVER (
            ORDER BY PassengerId
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) as moving_avg_3,
        ROUND(SUM(Fare) OVER (
            ORDER BY PassengerId
        ), 2) as cumulative_fare
    FROM titanic
    WHERE Pclass = 1
    ORDER BY PassengerId
    LIMIT 8
"""
    )
)

# Percent of total
print("\n--- Q11: Each passenger's fare as % of total fares ---")
print(
    sql(
        """
    SELECT 
        Name,
        Pclass,
        Fare,
        ROUND(Fare / SUM(Fare) OVER () * 100, 3) as pct_of_total,
        ROUND(Fare / SUM(Fare) OVER (PARTITION BY Pclass) * 100, 2) as pct_of_class
    FROM titanic
    ORDER BY Fare DESC
    LIMIT 8
"""
    )
)

      ADVANCED WINDOW FUNCTIONS

--- Q9: Divide passengers into fare quartiles ---
   fare_quartile  passengers  min_fare  max_fare  survival_pct
0              1         223      0.00      7.90          19.7
1              2         223      7.93     14.45          30.5
2              3         223     14.45     31.00          45.3
3              4         222     31.28    512.33          58.1

--- Q10: Running avg fare for 1st class passengers ---
   PassengerId      Fare  moving_avg_3  cumulative_fare
0            2   71.2833         71.28            71.28
1            4   53.1000         62.19           124.38
2            7   51.8625         58.75           176.25
3           12   26.5500         43.84           202.80
4           24   35.5000         37.97           238.30
5           28  263.0000        108.35           501.30
6           31   27.7208        108.74           529.02
7           32  146.5208        145.75           675.54

--- Q11: Each passenger's fare as % of to

## 7. Key Takeaways — Day 13 🎯

### Window Functions vs GROUP BY
- **GROUP BY** collapses rows — 891 rows → 3 rows
- **Window Functions** keep all rows — 891 rows → 891 rows
- Syntax: `function() OVER (PARTITION BY col ORDER BY col)`

### Ranking Functions
| Function | Ties | Gaps | Use When |
|---|---|---|---|
| ROW_NUMBER() | Unique always | No ties | Need exactly 1 row per group |
| RANK() | Same rank | Yes (skips) | Ties matter, gaps OK |
| DENSE_RANK() | Same rank | No gaps | Ties matter, no gaps wanted |

### LAG & LEAD
- `LAG(col, n)` — value from n rows BEFORE current row
- `LEAD(col, n)` — value from n rows AFTER current row
- First row LAG = NULL, Last row LEAD = NULL
- Used for: month-over-month comparisons, time between events

### Running Calculations
- `SUM() OVER (ORDER BY col)` — cumulative total
- `AVG() OVER (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` — moving average
- `NTILE(4) OVER (ORDER BY col)` — divide into quartiles
- `col / SUM(col) OVER ()` — percentage of total

### CTEs
- `WITH name AS (SELECT ...)` — named temporary result
- Chain multiple CTEs with commas
- Much cleaner than nested subqueries
- Can reference same CTE multiple times in one query

### Key Titanic Insights
- Top fare quartile: 58.1% survival vs bottom: 19.7% — 3x difference!
- Above-average payers within each class also survived more
- 1st class females avg fare: £106 vs 2nd class females: £22
- Three passengers each paid 1.785% of ALL ship revenue!

### Tools Used
- `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` — ranking
- `LAG(col) OVER (ORDER BY ...)` — previous row value
- `SUM(col) OVER (ORDER BY ... ROWS BETWEEN ...)` — running total
- `WITH cte AS (...)` — Common Table Expression
- `NTILE(n) OVER (ORDER BY ...)` — divide into n buckets